# Combine Run Results

This notebook walks through all subdirectories under a chosen root directory, reads `summary.json` and `history.csv` from each run folder, and writes combined CSV exports.

- `combined_run_summary.csv`: one row per run
- `combined_history_rows.csv`: one row per epoch/history row


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd


In [ ]:
# Set these paths before running.
RUNS_ROOT = Path('/home/pkunwar/characterize_ttlora/phases/2.1.ttlora_core_count_study/runs')
OUTPUT_DIR = Path('/home/pkunwar/characterize_ttlora/phases/2.2.different_dataset_models_rank/analysis_outputs/combined_runs_from_remote')

# If True, only collect folders that contain summary.json.
REQUIRE_SUMMARY = True

# If True, only collect folders that contain history.csv.
REQUIRE_HISTORY = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_ROOT, OUTPUT_DIR


In [ ]:
def _json_safe(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=True)
    return value


def flatten_dict(data: dict[str, Any], prefix: str = '') -> dict[str, Any]:
    flattened: dict[str, Any] = {}
    for key, value in data.items():
        full_key = f'{prefix}{key}' if prefix else key
        if isinstance(value, dict):
            flattened.update(flatten_dict(value, prefix=f'{full_key}.'))
        else:
            flattened[full_key] = _json_safe(value)
    return flattened


def read_summary(summary_path: Path) -> dict[str, Any]:
    with summary_path.open('r', encoding='utf-8') as handle:
        summary = json.load(handle)
    return flatten_dict(summary)


def read_history(history_path: Path) -> pd.DataFrame:
    return pd.read_csv(history_path)


def history_run_features(history_df: pd.DataFrame) -> dict[str, Any]:
    features: dict[str, Any] = {}
    if history_df.empty:
        return features

    last_row = history_df.iloc[-1].to_dict()
    for key, value in last_row.items():
        features[f'history_last.{key}'] = value

    if 'validation_perplexity' in history_df.columns:
        best_idx = history_df['validation_perplexity'].astype(float).idxmin()
        best_row = history_df.loc[best_idx].to_dict()
        for key, value in best_row.items():
            features[f'history_best.{key}'] = value

    features['history_rows'] = len(history_df)
    return features


def find_run_directories(root: Path) -> list[Path]:
    run_dirs: list[Path] = []
    for path in sorted(root.rglob('*')):
        if not path.is_dir():
            continue

        summary_path = path / 'summary.json'
        history_path = path / 'history.csv'

        has_summary = summary_path.exists()
        has_history = history_path.exists()

        if REQUIRE_SUMMARY and not has_summary:
            continue
        if REQUIRE_HISTORY and not has_history:
            continue
        if has_summary or has_history:
            run_dirs.append(path)
    return run_dirs


In [ ]:
run_dirs = find_run_directories(RUNS_ROOT)
print(f'Found {len(run_dirs)} run directories under {RUNS_ROOT}')
run_dirs[:10]


In [ ]:
run_summary_rows: list[dict[str, Any]] = []
history_rows: list[pd.DataFrame] = []
errors: list[dict[str, str]] = []

for run_dir in run_dirs:
    summary_path = run_dir / 'summary.json'
    history_path = run_dir / 'history.csv'

    row: dict[str, Any] = {
        'run_dir': str(run_dir),
        'run_name': run_dir.name,
        'has_summary': summary_path.exists(),
        'has_history': history_path.exists(),
    }

    try:
        if summary_path.exists():
            row.update(read_summary(summary_path))
    except Exception as exc:
        errors.append({'run_dir': str(run_dir), 'file': str(summary_path), 'error': repr(exc)})

    try:
        if history_path.exists():
            history_df = read_history(history_path)
            row.update(history_run_features(history_df))
            history_df = history_df.copy()
            history_df.insert(0, 'run_dir', str(run_dir))
            history_df.insert(1, 'run_name', run_dir.name)
            history_rows.append(history_df)
    except Exception as exc:
        errors.append({'run_dir': str(run_dir), 'file': str(history_path), 'error': repr(exc)})

    run_summary_rows.append(row)

combined_runs_df = pd.DataFrame(run_summary_rows)
combined_history_df = pd.concat(history_rows, ignore_index=True) if history_rows else pd.DataFrame()
errors_df = pd.DataFrame(errors)

print('combined_runs_df shape:', combined_runs_df.shape)
print('combined_history_df shape:', combined_history_df.shape)
print('errors_df shape:', errors_df.shape)


In [ ]:
combined_runs_path = OUTPUT_DIR / 'combined_run_summary.csv'
combined_history_path = OUTPUT_DIR / 'combined_history_rows.csv'
errors_path = OUTPUT_DIR / 'combine_errors.csv'

combined_runs_df.to_csv(combined_runs_path, index=False)
if not combined_history_df.empty:
    combined_history_df.to_csv(combined_history_path, index=False)
if not errors_df.empty:
    errors_df.to_csv(errors_path, index=False)

print('Wrote:', combined_runs_path)
print('Wrote:', combined_history_path if not combined_history_df.empty else 'no history rows to write')
print('Wrote:', errors_path if not errors_df.empty else 'no errors to write')


In [ ]:
display(combined_runs_df.head())
display(combined_runs_df.columns.tolist())


In [ ]:
if not combined_history_df.empty:
    display(combined_history_df.head())
else:
    print('No history rows were collected.')

if not errors_df.empty:
    display(errors_df)
else:
    print('No read errors.')
